# 🚀 [Colab 실습] YOLOv9 실전 — 학습부터 ONNX Export까지

**온디바이스 AI 프로그래밍 · Day 3 「YOLOv9 실시간 파이프라인」 표준 학습 실습**

| 항목 | 내용 |
| --- | --- |
| 위치 | 「YOLO v1 첫걸음」(원리) → **이 실습(진짜 v9 학습)** → 「yolov9-dpi 예제」(디퍼아이 컴파일·NPU 배포) |
| 도구 | ultralytics (YOLOv9 공식 지원) + PyTorch + ONNX |
| 환경 | **Google Colab GPU 런타임 (T4)** — 런타임 → 런타임 유형 변경 → T4 GPU |
| 진행 | 위에서부터 셀을 하나씩 실행 (`Shift + Enter`) · 학습 셀 ☕ 5~8분 1회 |

## 이 실습의 목표

첫걸음에서 손으로 만든 미니 YOLO의 모든 개념 — 그리드·책임·손실 3항·NMS·IoU — 이
**진짜 YOLOv9에서 어떤 모습으로 살아 있는지** 확인하며, **학습 → 평가(mAP) → 추론 실험 → ONNX Export**의
표준 파이프라인을 완주합니다. 마지막에 디퍼아이 `yolov9-dpi` 파이프라인과의 대응표로
NPU 배포(Day 3 본편)로 넘어갈 준비를 마칩니다.

## 로드맵

| Part | 주제 | 첫걸음·애니메이션 연결 |
| --- | --- | --- |
| 1 | 사전학습 v9 첫 만남 — 출력 텐서 해부 | 4×4 그리드 → 멀티스케일 3549셀 |
| 2 | 학습 — coco128 파인튜닝 (☕) | 손실 3항의 실전판 (box·cls·dfl) |
| 3 | 평가 — mAP50 / mAP50-95 해부 | IoU 애니메이션의 그 임계값 |
| 4 | 추론 실험 — conf·NMS 스윕 | NMS 애니메이션 실전판 |
| 5 | ONNX Export — NPU로 가는 여권 | ONNX 첫걸음·교안 함정 6가지 |
| 6 | 디퍼아이 yolov9-dpi 대응표 | Day 3 본편으로 |


---
# Part 0. 환경 준비 — GPU 확인과 설치

In [ ]:
# GPU 런타임인지 확인 (T4가 보여야 정상 — 안 보이면 런타임 유형을 GPU로!)
!nvidia-smi -L
import torch
DEV = 0 if torch.cuda.is_available() else "cpu"
print("학습 디바이스:", "GPU ✅" if DEV == 0 else "CPU (느립니다 — GPU 런타임 권장!)")

In [ ]:
%pip install -q ultralytics onnx onnxruntime
import ultralytics
ultralytics.checks()

In [ ]:
# 한글 폰트 설정 — 그래프 제목·라벨이 □□로 깨지지 않도록 (시리즈 공통 셀)
import matplotlib.pyplot as plt
try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"],
                   capture_output=True, timeout=120)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rc("font", family="NanumGothic")
    print("한글 폰트 설정 완료 ✅")
except Exception as e:
    print("한글 폰트 설정 생략(제목이 깨질 수 있음):", e)
plt.rc("axes", unicode_minus=False)

---
# Part 1. 사전학습 YOLOv9 첫 만남

### Step 1-1. 모델 로드 — GELAN 블록을 눈으로 확인

`yolov9t`(tiny)는 v9 가족의 막내입니다: **2.1M 파라미터, 4.7MB** (논문의 v9-C는 25.3M).
구조를 출력해 애니메이션에서 본 **GELAN**이 실제 블록 이름으로 박혀 있는 것을 확인합시다.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov9t.pt")                      # 사전학습(COCO) 가중치 자동 다운로드
n_params = sum(p.numel() for p in model.model.parameters())
print(f"YOLOv9-t: 파라미터 {n_params/1e6:.2f}M")
print()
# 구조에서 GELAN 흔적 찾기
import re
names = {type(m).__name__ for m in model.model.modules()}
gelan = sorted(n for n in names if "ELAN" in n)
print("모델 속 GELAN 계열 블록:", gelan)
print("→ 애니메이션 「YOLOv9 PGI·GELAN」의 그 블록들입니다 (RepNCSPELAN4 = GELAN 본체)")
print()
print("💡 PGI 보조 분기는? — 추론용 배포 가중치에는 이미 '걷어낸' 상태(비계 제거 완료).")
print("   학습 프레임워크 내부에서만 쓰이고, 우리가 받은 모델엔 흔적이 없습니다 — 추론 비용 +0의 증거.")

### Step 1-2. 첫 추론 — 사전학습의 힘

In [ ]:
import matplotlib.pyplot as plt

res = model.predict("https://ultralytics.com/images/bus.jpg",
                    imgsz=416, conf=0.25, verbose=False)[0]
plt.figure(figsize=(7, 8))
plt.imshow(res.plot()[..., ::-1])               # BGR→RGB
plt.axis("off"); plt.title(f"yolov9t 첫 추론 — 검출 {len(res.boxes)}개")
plt.show()

for b in res.boxes:
    print(f"  {res.names[int(b.cls)]:<10} conf {float(b.conf):.2f}  박스 {b.xyxy[0].int().tolist()}")

### Step 1-3. 출력 텐서 해부 — 첫걸음의 4×4가 어떻게 자랐나

원시 출력(NMS 전)의 shape를 직접 확인합니다. 첫걸음과의 대응이 핵심입니다.

In [ ]:
import numpy as np
mdev = next(model.model.parameters()).device      # predict() 후 모델이 GPU에 있을 수 있음
x = torch.zeros(1, 3, 416, 416, device=mdev)      # 입력을 모델과 같은 디바이스에 생성
with torch.no_grad():
    raw = model.model(x)[0].cpu()
print("원시 출력 shape:", tuple(raw.shape), " ← (배치, 4+클래스80, 셀 수)")
print()
print(f"셀 수 3549의 정체: 52² + 26² + 13² = {52**2 + 26**2 + 13**2}")
print(f"  416/8 = 52 (작은 물체 담당 · 촘촘한 그리드)")
print(f"  416/16 = 26 (중간)")
print(f"  416/32 = 13 (큰 물체 담당 · 성긴 그리드)")
print()
print("┌── 첫걸음 미니 YOLO ──────────┬── YOLOv9 ─────────────────────────┐")
print("│ 4×4 그리드 한 벌            │ 52²+26²+13² 멀티스케일 세 벌       │")
print("│ 셀 벡터 [x,y,w,h,conf,p×3]  │ 셀 벡터 [x,y,w,h, cls×80] (앵커프리)│")
print("│ conf 칸 별도                │ conf 없음 — 클래스 점수가 겸임      │")
print("│ 작은 물체의 저주(41%)       │ 52² 그리드가 그 저주의 처방          │")
print("└─────────────────────────────┴───────────────────────────────────┘")

> **✅ Part 1 확인**
> - [ ] GELAN 블록(RepNCSPELAN4)을 모델 구조에서 직접 찾았다
> - [ ] PGI 분기가 배포 가중치에 없는 이유(추론 비용 +0)를 설명할 수 있다
> - [ ] 3549 = 52²+26²+13² 멀티스케일이 "작은 물체의 저주"의 처방임을 이해했다

---
# Part 2. 학습 — coco128 파인튜닝

### Step 2-1. 데이터: coco128

**coco128**은 COCO의 축소판(128장, 80클래스)으로, 학습 파이프라인 실습의 표준 데이터셋입니다.
첫 실행 시 자동 다운로드(6.6MB)됩니다.

> ⚠️ **정직 고지**: coco128은 train과 val이 **같은 128장**입니다. 여기서 얻는 mAP는
> "파이프라인이 잘 도는지 + 모델이 배우고 있는지" 확인용이지, 일반화 성능이 아닙니다.
> (실전·리포트에선 train/val 분리된 데이터를 쓰세요 — Part 6 도전과제)

### Step 2-2. 학습 실행 (☕ T4 기준 5~8분, 1회만)

`imgsz=416`은 **디퍼아이 파이프라인 정합**을 위한 선택입니다 — `yolov9-dpi` 예제의
학습 옵션이 `--img 416`이고, TACHY 컴파일 입력이 256×416/416×416이기 때문입니다.

In [ ]:
# ⚠️ 학습 직전 모델을 '새로' 로드합니다 — 중요!
# Part 1의 predict()는 추론 최적화를 위해 모델을 fuse(Conv+BN 합치기)합니다.
# fuse된 모델로 train을 시작하면 가중치 이름이 어긋나 사전학습 이식이
# 150/1339개만 되어(로그로 확인 가능) 사실상 밑바닥 학습이 되어버립니다.
model = YOLO("yolov9t.pt")

results = model.train(
    data="coco128.yaml",     # 자동 다운로드
    epochs=30,
    imgsz=416,               # 디퍼아이 yolov9-dpi와 동일 (--img 416)
    batch=16,
    device=DEV,
    name="v9_coco128",
    plots=True,
)
print("학습 완료! 산출물 폴더:", results.save_dir)

> 🔎 **학습이 제대로 시작됐는지 확인하는 법** — 위 셀 로그 앞부분에서
> `Transferred 1339/1339 items from pretrained weights` 를 찾으세요.
> **1339/1339**여야 정상(사전학습 전체 이식)이고, `150/1339`처럼 나오면 fuse된 모델로
> 학습이 시작된 것입니다(이 경우 mAP가 0 근처에 머뭅니다). 30에폭 후 mAP50이
> 대략 0.5~0.7 수준이면 파인튜닝이 잘 된 것입니다.

### Step 2-3. 학습 곡선 읽기 — 손실 3항의 실전판

첫걸음의 손실 3항(좌표 / conf / 클래스)이 v9에선 이렇게 바뀌었습니다:

| 첫걸음 미니 YOLO | YOLOv9 | 의미 |
| --- | --- | --- |
| 좌표 제곱합 (λ=5) | **box_loss** (CIoU) | 박스를 IoU 기반으로 직접 최적화 |
| — | **dfl_loss** (Distribution Focal) | 박스 경계를 "분포"로 회귀 — 앵커 프리의 크기 학습 무기 |
| conf + 클래스 | **cls_loss** (BCE) | 클래스 점수가 conf 역할까지 겸임 |

In [ ]:
from IPython.display import Image, display
import os
display(Image(os.path.join(results.save_dir, "results.png"), width=980))
print("읽는 법: 왼쪽 3칸(train 손실)과 그 옆(val 손실)이 함께 내려가고,")
print("        오른쪽 mAP50 / mAP50-95가 올라가면 — 3항이 협업 중이라는 뜻입니다.")

> **✅ Part 2 확인**
> - [ ] imgsz=416의 이유(디퍼아이 정합)를 말할 수 있다
> - [ ] box/dfl/cls 3항을 첫걸음의 3항과 대응시킬 수 있다
> - [ ] coco128 mAP의 한계(train=val)를 인지했다

---
# Part 3. 평가 — mAP 해부

### Step 3-1. mAP50 vs mAP50-95 — IoU 임계값의 귀환

- **mAP50**: 예측이 정답과 **IoU > 0.5**면 성공으로 치고 계산한 평균 정밀도 — IoU 애니메이션의 그 "합격선"
- **mAP50-95**: 임계값을 0.5, 0.55, ..., 0.95로 **10단계 올려가며** 평균 — 박스가 얼마나 "정확히" 맞는지까지 요구
- 항상 mAP50-95 ≤ mAP50: 자가 빡빡해질수록 점수는 내려갑니다

In [ ]:
best = YOLO(os.path.join(results.save_dir, "weights/best.pt"))
metrics = best.val(data="coco128.yaml", imgsz=416, device=DEV, verbose=False)

print(f"mAP50    = {metrics.box.map50:.3f}   (IoU>0.5 합격선)")
print(f"mAP50-95 = {metrics.box.map:.3f}   (0.5~0.95 평균 — 더 빡빡)")
print(f"차이가 곧 '박스 정밀도의 여지'입니다.")
print()
# 클래스별 AP 상위/하위
pairs = sorted(zip(metrics.names.values(), metrics.box.ap50), key=lambda x: -x[1])
print("AP50 상위 5:", [(n, round(float(a), 2)) for n, a in pairs[:5]])
print("AP50 하위 5:", [(n, round(float(a), 2)) for n, a in pairs[-5:]])
print("→ 하위 클래스의 공통점을 관찰해 보세요 (작거나, 드물거나, 겹쳐 있거나)")

### Step 3-2. PR 곡선과 혼동행렬 — 그림으로 보는 성적표

In [ ]:
val_dir = metrics.save_dir
display(Image(os.path.join(val_dir, "BoxPR_curve.png"), width=560))
display(Image(os.path.join(val_dir, "confusion_matrix_normalized.png"), width=560))
print("PR 곡선: conf 임계를 훑으며 그린 정밀도-재현율 — 곡선 아래 면적이 AP입니다.")
print("혼동행렬: 어떤 클래스를 어떤 클래스로 착각하는지 — background 열이 미검출입니다.")

> **✅ Part 3 확인**
> - [ ] mAP50과 mAP50-95의 차이를 IoU 임계로 설명할 수 있다
> - [ ] PR 곡선의 면적 = AP 관계를 이해했다
> - [ ] 혼동행렬에서 background(미검출) 열을 읽을 수 있다

---
# Part 4. 추론 파라미터 실험 — conf와 NMS를 실전에서 돌려보기

첫걸음에서 10줄로 구현한 그 두 개의 손잡이가, 실전에선 `predict()`의 인자입니다:
- `conf` — 신뢰도 임계 (낮추면 후보 ↑, 오검출 ↑)
- `iou` — **NMS 억제 임계** (높이면 겹친 박스 관대 → 중복 ↑)

### Step 4-1. conf 스윕

In [ ]:
URL = "https://ultralytics.com/images/bus.jpg"
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, c in zip(axes, [0.10, 0.25, 0.60]):
    r = best.predict(URL, imgsz=416, conf=c, iou=0.5, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
    ax.set_title(f"conf={c} → {len(r.boxes)}개")
plt.suptitle("conf 임계 스윕 — 낮출수록 후보가 늘고, 오검출도 늘어난다")
plt.tight_layout(); plt.show()

### Step 4-2. NMS(iou) 스윕 — 애니메이션의 ✂를 실전에서

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, t in zip(axes, [0.30, 0.50, 0.90]):
    r = best.predict(URL, imgsz=416, conf=0.20, iou=t, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
    ax.set_title(f"NMS iou={t} → {len(r.boxes)}개")
plt.suptitle("NMS 임계 스윕 — 0.9로 올리면 겹친 중복 박스가 살아남는다 (✂가 무뎌짐)")
plt.tight_layout(); plt.show()
print("💡 첫걸음 NMS 10줄과 「책임 셀·NMS」 애니메이션의 그 IoU>임계 → ✂ 규칙이")
print("   그대로 동작 중입니다. NPU 배포 시 이 후처리는 CPU에서 분리 실행됩니다 (Day 3).")

> **✅ Part 4 확인**
> - [ ] conf를 낮추면 박스 수가 어떻게 변하는지 관찰했다
> - [ ] NMS iou=0.9에서 중복 박스가 살아남는 것을 확인했다

---
# Part 5. ONNX Export — NPU로 가는 여권

「ONNX Export 첫걸음」에서 밑바닥부터 만든 그 과정 — **트레이스 → 설계도 → 검증** — 을
진짜 도구로 실행합니다. 교안 체크리스트 그대로:

| 교안 체크 | export 인자 |
| --- | --- |
| **고정 shape** (dynamic_axes 금지) | `dynamic=False`, `imgsz=416` |
| **opset ≥ 13** | `opset=13` |
| **단순화** (Reshape 난무 청소) | `simplify=True` (onnxslim) |

### Step 5-1. Export 실행

In [ ]:
onnx_path = best.export(format="onnx", imgsz=416, opset=13,
                        simplify=True, dynamic=False)
import os
print(f"생성: {onnx_path} ({os.path.getsize(onnx_path)/1e6:.1f}MB)")

### Step 5-2. 설계도 검사 — 여권 심사

In [ ]:
import onnx
m = onnx.load(onnx_path)
inp = m.graph.input[0]
dims = [d.dim_value for d in inp.type.tensor_type.shape.dim]
ops = {n.op_type for n in m.graph.node}

print(f"입력 shape : {dims}   ← 배치까지 상수 1로 박제 (고정 shape ✔)")
print(f"opset      : {m.opset_import[0].version}   (≥13 ✔)")
print(f"노드 수    : {len(m.graph.node)}")
print(f"연산자 종류: {len(ops)}종 — 예: {sorted(ops)[:8]} ...")
print()
print("💡 첫걸음의 '설계도 딕셔너리'가 protobuf로 저장된 것 — Netron(netron.app)에")
print("   이 파일을 끌어다 놓으면 그래프를 눈으로 볼 수 있습니다.")

### Step 5-3. 국경 반대편 검증 — PyTorch vs onnxruntime

첫걸음의 클라이맥스("파일만 읽어 재실행 → 같은 답")를 실전 규모로 반복합니다.

In [ ]:
import onnxruntime as ort
import numpy as np, torch

x = np.random.rand(1, 3, 416, 416).astype(np.float32)
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
y_onnx = sess.run(None, {sess.get_inputs()[0].name: x})[0]

cpu_model = YOLO(os.path.join(results.save_dir, "weights/best.pt"))
with torch.no_grad():
    y_pt = cpu_model.model(torch.from_numpy(x))[0].numpy()

diff = np.abs(y_pt - y_onnx).max()
print(f"PyTorch 출력 {y_pt.shape}  vs  ONNX 출력 {y_onnx.shape}")
print(f"최대 오차: {diff:.2e} → 일치(atol 1e-3): {np.allclose(y_pt, y_onnx, atol=1e-3)}")
print()
print("✅ 모델이 프레임워크 국경을 넘었습니다. 오차 ~1e-3은 fp32 연산 순서 차이 수준.")
print("   실전 다음 주자: onnxruntime 대신 디퍼아이 컴파일러가 이 파일을 받습니다.")

> **✅ Part 5 확인**
> - [ ] 체크리스트 3종(고정 shape·opset 13·simplify)을 인자로 지정했다
> - [ ] onnx.load로 입력 [1,3,416,416]과 opset을 직접 확인했다
> - [ ] PyTorch↔ONNX 출력 일치를 수치로 검증했다

---
# Part 6. 디퍼아이 yolov9-dpi 파이프라인으로 — 대응표

오늘 실습과 Day 3 본편(`yolov9_dpi_example.ipynb`)의 관계입니다.
**같은 뼈대에, NPU 전용 단계가 추가**됩니다:

| 오늘 (표준 파이프라인) | yolov9-dpi (디퍼아이·NPU) | 비고 |
| --- | --- | --- |
| `yolov9t` (2.1M) | **bsnet-t** (`models/deeper-i/bsnet-t.yaml`) | BlackSwan 전용 v9 계열 백본 |
| `coco128.yaml` | `data/{target}.yaml` (커스텀) | 형식은 동일한 YOLO yaml |
| `model.train(imgsz=416)` | `train.py --img 416` (1단계: 기본 학습) | 옵션 철학 동일 |
| *(해당 없음)* | **2단계: bsnet-t-o + hyp.optimize** | NPU 최적화 학습 — QAT·XWN의 실전 (교안 Day 2) |
| `export(opset=13, ...)` | `compile/compile_linux.py`가 일괄 수행 | `ONNX_INPUT_SHAPE = "1 3 256 416"` |
| onnxruntime 검증 | **TACHY 컴파일 → `model_*.tachyrt`** | `fc.out`, `block_4bit.out` 필요 |
| `predict(conf, iou)` | Tachy-Shield 추론 + **NMS는 CPU 분리** | `post_process_256x416.json` |

컴파일 6단계에서의 위치:

```
[①최적화 학습(QAT·XWN)] → [②Export] → [③검사·단순화] → [④디퍼아이 컴파일러] → [⑤바이너리] → [⑥Tachy-Shield 배포]
 오늘: ①의 기본 학습 + ②③ 완주      dpi 예제: ①의 2단계 최적화 + ④⑤⑥
```

### 리포트 과제 — 직접 실험하고 표를 채우세요

| 실험 | 조건 | mAP50 | mAP50-95 | 관찰 |
| --- | --- | --- | --- | --- |
| 기준 | epochs=30, imgsz=416, yolov9t | | | |
| 실험1 | epochs 10 / 60 비교 | | | |
| 실험2 | imgsz 320으로 학습·평가 | | | |
| 실험3 | `yolov9s`로 교체 | | | |

**분석 질문 (2~3문장씩):**
1. 실험1에서 epochs를 늘릴 때 train 손실과 mAP의 추세가 다르게 움직이나요? coco128의 train=val 특성과 연결해 설명하세요.
2. 실험2(320)에서 어떤 크기의 물체가 더 손해를 볼 것으로 예상되고, 실제로 그런가요? (힌트: 첫걸음 "작은 물체의 저주" + 멀티스케일 그리드 크기 변화 320/8=40 ...)
3. 실험3에서 파라미터·mAP·추론 속도(val 출력의 Speed)를 비교하고, "NPU 20+ FPS 목표"라면 어느 모델을 고를지 근거를 쓰세요.

### ✏️ 심화 도전 과제 (선택)

1. **커스텀 데이터 학습**: Roboflow 등에서 train/val 분리된 소형 데이터셋(YOLO 형식)을 받아
   동일 파이프라인으로 학습 — coco128의 한계를 벗어난 진짜 mAP 측정
2. **ONNX 단독 추론기**: onnxruntime + 첫걸음의 NMS 10줄로, ultralytics 없이
   `bus.jpg → 박스` 전 과정을 직접 구현 (전처리 letterbox 포함) — Day 3 `post_process.py`의 미니어처
3. **INT8로 한 걸음**: `export(format="onnx", half=True)`(FP16) 후 크기·오차 비교 —
   양자화 첫걸음의 감각으로 관찰
4. **yolov9-dpi 완주**: 본편 매뉴얼 따라 bsnet-t 2단계 학습 → `.tachyrt` 컴파일 →
   Tachy-Shield 실기 배포 (fc.out·block_4bit.out 배치 잊지 말 것)

---

수고하셨습니다! 🎉 이제 여러분은 YOLOv9를 **원리(첫걸음) → 학습(오늘) → 배포(dpi)**의
세 층위로 다룰 수 있습니다. 오늘 만든 `best.pt`와 `best.onnx`, 그리고 416이라는 숫자가
Day 3 본편에서 그대로 이어집니다.
